# Open problems

There are many open problems related to Still Lifes to work on, both within and outside this repository. In this Notebook we list a few, hoping these may trigger your enthusiasm for further exploration.

## Overview

1. ~~Create a Tile database for level over 5~~ ✅ Resolved — see below and `tile_generation_sat.ipynb`
2. 

## 1. Tiles with level over 5 — resolved ✅

This used to read: *"When using the existing Gurobi algorithm, it takes quite a while to find all Tiles of level 5. It takes a much longer time to find those of level 6 (over two days running locally), and it is probably impossible to find those of level 7. This is probably not because the task is too difficult, but because the algorithm is not designed sufficiently cleverly. This needs attention."*

The algorithm has been redesigned. A SAT-based formulation over the free symmetry orbits (`gol_mosaics.sat_search`, pipeline in `workstation/level6_search/`) enumerates **all 332,321 level-6 Tiles in about two seconds** on a laptop — no Gurobi licence required — validated byte-exactly against the shipped level 3–5 databases, by an exhaustive SAT-free brute force at level 4, across two unrelated solvers, and by whole-Mosaic stability checks. Level 6 now ships with the package (`PatternLibrary.load(6)`, 2.7 MB packed) and is available in the web app. The formerly hand-made dead-edge lists are now *derived* from the Mosaic geometry, which also unlocks level 7 (~10⁸ expected Tiles; the pipeline supports it with packed storage and checkpointing). The full story, including the validation ladder and the generalisation to other Life-like rules, is in `tile_generation_sat.ipynb`.

## Recently resolved

These issues from earlier versions of this list have since been addressed in the codebase:

- **Tile database for level over 5** ✅ Resolved — see section 1 above and `tile_generation_sat.ipynb`. All 332,321 level-6 Tiles enumerated exhaustively via SAT in ~2 s; level 6 ships with the package and the web app; dead edges are now derived from the Mosaic geometry for any level.

- **Randomise values by default** ✅ Implemented. `MosaicGenerator.__init__` now auto-selects `grid_size`, `level` and `eca_rule` from sensible ranges via `_auto_select_grid_size`, `_auto_select_level` and `_auto_select_eca_rule` whenever they are not supplied.

- **Remove background** ✅ Implemented properly. `ImageProcessor` now removes the background with the optional `rembg` package, including an `'auto'` mode (driven by `has_background()`) and hardware-accelerated ONNX Runtime providers. This supersedes the earlier naive approach.

- **Simple export** ✅ Implemented. `gol_mosaics.export.GollyExporter` exports a Still Life to a `.cells` file for direct import into Golly (and also to `.rle`), with an optional glider in any corner.

- **Intelligent colour choice** ✅ Implemented. Dark cells on a light background are enforced via the `dark_on_light` option in `ColorScheme.warhol`. (It could still be coded more pythonically / generalised beyond the Warhol scheme.)

- **Automatic supersample choice** ✅ Fixed. `ECABackground.generate` now builds the pattern at `ceil(dim/supersample)` low resolution, upsamples with `np.repeat`, and crops back to the exact `(height, width)`, so the supersample no longer has to divide the dimensions and never raises. The auto-selection (now inlined in `generate_from_pil`) therefore uses the target cell size (~15 px) directly instead of the nearest divisor. This fixes both the old "results in an error" bug (the auto-selected supersample divided the mosaic width but not its different height) and the "valid options too far from 15" quality complaint, and merges the former duplicate "Auto select supersample" item.

- **Image input validation** ✅ Fixed. A removed background that circled the entire image used to result in everything being removed: `binary_fill_holes` in `MosaicGenerator._build_mask` treated a subject touching no image edge as one giant enclosed hole and converted it to background, so the ECA overlay covered the whole mosaic. `_build_mask` now uses a size-thresholded fill (`_fill_small_holes`): only holes smaller than a quarter Tile — the tiny inter-Tile gaps of a few pixels — are filled, while enclosed subjects stay foreground. Edge-touching images produce byte-identical masks, and regression tests cover both cases. This also settles the former "Handle more input image types" alternative, which proposed working around the same `binary_fill_holes` behaviour.

## Still open

- **Level-7 Tile database** The blocker is gone (dead edges derived; pipeline ready with packed storage and checkpointing), but the run itself is pending: ~10⁸ expected Tiles, hours of solving on a multi-core machine, and a few GB of packed output. `python search.py run --level 7 --cube-bits 16` in `workstation/level6_search/`.

- **Necessity of the dead edges** The derived dead edges are provably *sufficient* for arbitrary Tile mixing, and the teeth test shows some are necessary — but a minimality study (which subsets of dead-edge cells could be freed for *some* pairings?) would sharpen the tiling theory and possibly enlarge the Tile sets.

- **Non-Conway Mosaics end to end** Tiles for other Life-like rules can now be enumerated (`sat_search.build_cnf(level, birth=..., survival=...)`, demonstrated for HighLife), but the ECA background and renderer pipeline have not been exercised with them.

- **The outer rim / odd edge behaviour** The mosaic uses the raw original pipeline: the rotation/padding rim simply inherits the Still Life background colour (the forced-rim footprint mask and `rim_color` overlay were removed). The edge of the Mosaic still does not always render cleanly — it can show too few Tiles or a cropped half-Tile. **This is not yet fully fixed and should be addressed again.**

- **grid_size parameter** is a bit ambiguous right now. An even-value check was added (odd `grid_size` now raises `ValueError`), but certain values still result in an image that is cut off in a surprising way.

- **UK English** Mostly done. All prose is now British English — docstrings, comments, Markdown and user-facing strings use spellings such as `colour`, `greyscale`, `neighbours` and `licence`. The US spellings that remain are deliberate: code identifiers (`colors.py`, `ColorScheme`, `color_scheme`) and third-party API names (matplotlib's `cmap='gray'`, PIL's `color=`, `gr.ColorPicker`) are kept US-spelled so the public API stays intact. Renaming those identifiers too would be the next, breaking step toward full consistency.

- **Ponds, Tiles, Mosaics** Be more consistent with naming these patterns.

- **Canvas pixels** When creating a canvas (like the Marilyn Diptych) it is no longer easy to map a single cell to a single pixel (because generally the Mosaics will have different grid sizes). This needs an elegant fix. That's not going to be easy.

- **.cells file downloadable in web app** There should be an option to download the .cells files in the web app, which can in turn be used to upload to Golly (to verify that it is indeed a still life)